In [3]:
import pandas as pd
import numpy as np
import nltk
from nltk.tokenize import word_tokenize, sent_tokenize
from nltk.corpus import stopwords
from collections import Counter

# Download NLTK data
nltk.download('punkt')
nltk.download('stopwords')

# Load the dataset (assuming the dataset is similar to what was provided earlier)
df = pd.read_csv('datasets/converted_nlp_dataset.csv')

# Define stopwords
stop_words = set(stopwords.words('english'))

# Function to compute stylometric features from text
def compute_stylometric_features(text):
    tokens = word_tokenize(text)
    words = [word for word in tokens if word.isalpha()]  # Keep only alphabetic tokens
    words_no_stopwords = [word for word in words if word.lower() not in stop_words]
    
    # Feature 1: Average word length
    avg_word_length = np.mean([len(word) for word in words]) if words else 0
    
    # Feature 2: Lexical diversity (unique words / total words)
    lexical_diversity = len(set(words)) / len(words) if words else 0
    
    # Feature 3: Hapax legomena (words that appear only once)
    word_freq = Counter(words)
    hapax_legomena_count = sum(1 for word, count in word_freq.items() if count == 1)
    hapax_legomena_ratio = hapax_legomena_count / len(words) if words else 0
    
    # Feature 4: Sentence length variability (average sentence length in words)
    sentences = sent_tokenize(text)
    sentence_lengths = [len(word_tokenize(sent)) for sent in sentences]
    avg_sentence_length = np.mean(sentence_lengths) if sentences else 0
    
    # Feature 5: POS tag distribution (noun, verb, adj ratios)
    pos_tags = nltk.pos_tag(words)
    pos_counts = Counter(tag for word, tag in pos_tags)
    
    noun_ratio = pos_counts['NN'] / len(pos_tags) if 'NN' in pos_counts else 0
    verb_ratio = pos_counts['VB'] / len(pos_tags) if 'VB' in pos_counts else 0
    adj_ratio = pos_counts['JJ'] / len(pos_tags) if 'JJ' in pos_counts else 0
    
    return {
        'avg_word_length': avg_word_length,
        'lexical_diversity': lexical_diversity,
        'hapax_legomena_ratio': hapax_legomena_ratio,
        'avg_sentence_length': avg_sentence_length,
        'noun_ratio': noun_ratio,
        'verb_ratio': verb_ratio,
        'adj_ratio': adj_ratio
    }

# Add stylometric features to the dataset (if not already present)
def add_stylometric_features_to_dataset(df):
    df['avg_word_length'] = df['raw_text'].apply(lambda x: compute_stylometric_features(x)['avg_word_length'])
    df['lexical_diversity'] = df['raw_text'].apply(lambda x: compute_stylometric_features(x)['lexical_diversity'])
    df['hapax_legomena_ratio'] = df['raw_text'].apply(lambda x: compute_stylometric_features(x)['hapax_legomena_ratio'])
    df['avg_sentence_length'] = df['raw_text'].apply(lambda x: compute_stylometric_features(x)['avg_sentence_length'])
    df['noun_ratio'] = df['raw_text'].apply(lambda x: compute_stylometric_features(x)['noun_ratio'])
    df['verb_ratio'] = df['raw_text'].apply(lambda x: compute_stylometric_features(x)['verb_ratio'])
    df['adj_ratio'] = df['raw_text'].apply(lambda x: compute_stylometric_features(x)['adj_ratio'])
    return df

# Update dataset with new stylometric features
df = add_stylometric_features_to_dataset(df)

# Update the authentication function to display a table of features that passed and failed
def authenticate_user(df, username, user_input, tolerance=0.2):
    # Check if the user exists in the dataset
    user_data = df[df['user_id'] == username]
    if user_data.empty:
        print(f"Access denied: User '{username}' not found.")
        return
    
    # Compute stylometric features of the user's input
    user_input_features = compute_stylometric_features(user_input)
    
    # Calculate the average of the user's stored features
    avg_features = user_data[['avg_word_length', 'lexical_diversity', 'hapax_legomena_ratio', 
                              'avg_sentence_length', 'noun_ratio', 'verb_ratio', 'adj_ratio']].mean()
    
    # Store the results for features within and outside tolerance
    feature_comparison = {'Feature': [], 'Stored Average': [], 'User Input': [], 'Status': []}

    # Compare features and classify whether they are within tolerance
    for feature in user_input_features:
        user_value = user_input_features[feature]
        avg_value = avg_features[feature]
        if abs(user_value - avg_value) > tolerance:
            feature_comparison['Feature'].append(feature)
            feature_comparison['Stored Average'].append(avg_value)
            feature_comparison['User Input'].append(user_value)
            feature_comparison['Status'].append('Outside Limit')
        else:
            feature_comparison['Feature'].append(feature)
            feature_comparison['Stored Average'].append(avg_value)
            feature_comparison['User Input'].append(user_value)
            feature_comparison['Status'].append('Within Limit')

    # Convert the comparison results to a DataFrame for better readability
    comparison_df = pd.DataFrame(feature_comparison)
    
    # Display the comparison table in Jupyter Notebook
    display(comparison_df)

    # Determine the overall status based on the results
    if any(comparison_df['Status'] == 'Outside Limit'):
        print(f"Access denied: Some features are outside the tolerance limits.")
    else:
        print(f"Access granted: User '{username}' authenticated successfully.")

# Main function to prompt user and authenticate
def main():
    # Step 1: Prompt the user for username and their input
    username = input("Enter your username: ")
    user_input = input("Enter your prompt: ")
    
    # Step 2: Authenticate the user based on their stylometric features and print the result table
    authenticate_user(df, username, user_input)

# Execute the main function
main()


[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\conmy\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\conmy\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


,Feature,Stored Average,User Input,Status
0,avg_word_length,4.943444,3.411765,Outside Limit
1,lexical_diversity,0.973864,0.735294,Outside Limit
2,hapax_legomena_ratio,0.947727,0.588235,Outside Limit
3,avg_sentence_length,12.150000,13.333333,Outside Limit
4,noun_ratio,0.237920,0.205882,Within Limit
5,verb_ratio,0.073129,0.000000,Within Limit
6,adj_ratio,0.113636,0.029412,Within Limit


Access denied: Some features are outside the tolerance limits.
